# Претрейн латентной диффузии: метрики и инференс

Ноутбук смотрит на запуск `train_ldm.py`: графики из `metrics.csv`, сетки сэмплов, которые обучение сохраняло по ходу,
и генерацию из EMA-весов с разбором по шагам расшумления, в латенте и после декодера VAE.

Ячейки можно перезапускать, пока идет обучение, они перечитывают файлы из папки запуска. Генерации нужен GPU.

In [ ]:
import csv
import io
import json
import math
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from diffusers import AutoencoderKL
from IPython.display import display
from matplotlib import colormaps
from PIL import Image

from models import UNet
from train import VPSDE

RUN_DIR = Path('/home/CCP/semenchuk.aa/DiffusionLearn/runs/ldm_v2')   # ldm_v1: первый запуск, со срывом на шаге 5400
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = json.loads((RUN_DIR / 'config.json').read_text())
print(f"шагов {cfg['steps']}, батч {cfg['batch']} x {cfg['devices']} GPU, pred {cfg['pred']}, base {cfg['base']}, "
      f"голова {cfg.get('head_ch') or 'одна'}, qk-norm {cfg.get('qk_norm', False)}, "
      f"lr {cfg['lr']} -> {cfg['lr_min']}, прогрев {cfg.get('warmup', 0)}, VAE {cfg['vae']}, scale {cfg['scale']:.5f}")

## Метрики

Лосс здесь eps-MSE без весов, как в `train.py`: при `v` он пересчитан в ошибку шума, поэтому сравним с прошлыми запусками.
Интервалы t: при малом t модель дорисовывает мелкие детали, при большом угадывает общую композицию.
Пунктир отмечает перезапуск с последнего чекпоинта.

Нижний ряд предупреждает о срыве раньше лосса: норма градиента до обрезки (обрезка на 1) и худший блок внимания.
Энтропия внимания в долях от log N: около 1 внимание размазано по всей карте, около 0 смотрит в одну точку.
В ldm_v1 перед срывом энтропия на 16x16 была 0.01 при логитах до 18 000.

In [ ]:
def read_metrics(path):
    """metrics.csv -> {колонка: np.array} и шаги, с которых обучение продолжалось после остановки.
    После перезапуска шаги с последнего чекпоинта пишутся заново, берем последнюю запись каждого шага."""
    with open(path) as f:
        rows = list(csv.DictReader(f))
    restarts = [int(r['step']) for prev, r in zip(rows, rows[1:]) if int(r['step']) <= int(prev['step'])]
    by_step = {int(r['step']): r for r in rows}
    rows = [by_step[s] for s in sorted(by_step)]
    return {k: np.array([float(r[k]) for r in rows]) for k in rows[0]}, restarts


def smooth(y, k=0.9):
    """Экспоненциальное сглаживание, как в TensorBoard."""
    out, acc = np.empty_like(y), y[0]
    for i, v in enumerate(y):
        acc = k * acc + (1 - k) * v
        out[i] = acc
    return out


log, restarts = read_metrics(RUN_DIR / 'metrics.csv')
s = log['step']
bins = ['loss_t0-0.2', 'loss_t0.2-0.4', 'loss_t0.4-0.6', 'loss_t0.6-0.8', 'loss_t0.8-1']

extra = 'grad_norm' in log                     # в ldm_v1 этих колонок еще не было
fig, axs = plt.subplots(3 if extra else 2, 2, figsize=(16, 15 if extra else 10))
for ax in axs.flat:
    ax.grid(alpha=0.5, zorder=1, which='both')
    ax.set_xlabel('step')
    for r in restarts:
        ax.axvline(r, color='gray', ls=':', zorder=1)

# 1. общий лосс
axs[0, 0].semilogy(s, log['loss'], color='C0', alpha=0.3, zorder=2)
axs[0, 0].semilogy(s, smooth(log['loss']), color='C0', zorder=2)
axs[0, 0].set_title('eps-MSE, среднее по t (бледная линия без сглаживания)')

# 2. лосс по интервалам t
for i, b in enumerate(bins):
    axs[0, 1].semilogy(s, smooth(log[b]), color=f'C{i}', label='t ' + b.replace('loss_t', ''), zorder=2)
axs[0, 1].set_title('eps-MSE по интервалам t, сглажено')
axs[0, 1].legend()

# 3. learning rate
axs[1, 0].semilogy(s, log['lr'], zorder=2)
axs[1, 0].set_title('learning rate')

# 4. скорость
axs[1, 1].plot(s, log['it_per_s'], zorder=2)
axs[1, 1].set_title('шагов в секунду')
axs[1, 1].set_ylim(0, None)

if extra:
    # 5. норма градиента до обрезки: среднее и максимум за интервал лога
    axs[2, 0].semilogy(s, log['grad_norm'], label='среднее', zorder=2)
    axs[2, 0].semilogy(s, log['grad_norm_max'], label='максимум', alpha=0.6, zorder=2)
    axs[2, 0].axhline(cfg['grad_clip'], color='k', ls='--', label='порог обрезки', zorder=2)
    axs[2, 0].set_title('норма градиента до обрезки')
    axs[2, 0].legend()

    # 6. худший блок внимания
    ax = axs[2, 1]
    ax.plot(s, log['attn_min_entropy'], color='C3', zorder=2)
    ax.set_ylabel('минимальная энтропия', color='C3')
    ax.set_ylim(0, 1)
    ax2 = ax.twinx()
    ax2.semilogy(s, log['attn_max_logit'], color='C0', zorder=2)
    ax2.set_ylabel('максимальный логит', color='C0')
    ax.set_title('худший блок внимания')

plt.tight_layout()
plt.show()

left = (cfg['steps'] - s[-1]) / max(log['it_per_s'][-1], 1e-9) / 3600
print(f"последний шаг {int(s[-1])} из {cfg['steps']}, лосс {smooth(log['loss'])[-1]:.4f}, осталось ~{left:.1f} ч")

In [ ]:
# энтропия и максимум логитов каждого блока внимания (attn.csv, есть с ldm_v2)
if (RUN_DIR / 'attn.csv').exists():
    att, _ = read_metrics(RUN_DIR / 'attn.csv')
    layers = [k[:-4] for k in att if k.endswith('_ent')]
    fig, axs = plt.subplots(1, 2, figsize=(16, 5.5))
    for i, n in enumerate(layers):
        c = plt.cm.viridis(i / max(len(layers) - 1, 1))
        axs[0].plot(att['step'], att[n + '_ent'], color=c, label=n, zorder=2)
        axs[1].semilogy(att['step'], att[n + '_max'], color=c, label=n, zorder=2)
    axs[0].set_title('энтропия внимания по блокам, доля от log N')
    axs[0].set_ylim(0, 1)
    axs[1].set_title('максимальный логит по блокам')
    for ax in axs:
        ax.grid(alpha=0.5, zorder=1, which='both')
        ax.set_xlabel('step')
    axs[1].legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()
else:
    print('attn.csv нет: в этом запуске внимание не логировалось')

## Сэмплы, которые обучение сохраняло по ходу

Каждые 2500 шагов: 16 картинок из одного и того же шума, EMA-веса, Heun 50 шагов.
В конце обучения добавляется сетка `final_sde1000`: те же 16 шумов, SDE 1000 шагов.

In [ ]:
WIDTH = 768    # сторона сетки на экране, в пикселях

grids = sorted((RUN_DIR / 'samples').glob('step_*.png')) + sorted((RUN_DIR / 'samples').glob('final_*.png'))
if grids:
    pngs = [p.read_bytes() for p in grids]
    img = widgets.Image(value=pngs[-1], format='png', layout=widgets.Layout(width=f'{WIDTH}px', height=f'{WIDTH}px'))
    info = widgets.HTML()
    play = widgets.Play(min=0, max=len(grids) - 1, value=len(grids) - 1, interval=800, description='запуск')
    slider = widgets.IntSlider(min=0, max=len(grids) - 1, value=len(grids) - 1, description='сетка',
                               layout=widgets.Layout(width=f'{WIDTH - 80}px'))
    widgets.jslink((play, 'value'), (slider, 'value'))

    def show_grid(i):
        img.value = pngs[i]
        info.value = f'<b>{grids[i].stem}</b>, {i + 1} из {len(grids)}'

    slider.observe(lambda change: show_grid(change['new']), names='value')
    show_grid(len(grids) - 1)
    display(widgets.VBox([widgets.HBox([play, slider]), info, img]))
else:
    print('сэмплов пока нет, первые появятся на шаге 2500')

In [ ]:
# первая строка каждой сетки: одни и те же 4 шума на каждом шаге обучения
ROW = 2 + 512 + 2   # отступ + картинка 512x512 + отступ

rows = [np.asarray(Image.open(p).reduce(2))[:ROW // 2] for p in grids]
if rows:
    fig, ax = plt.subplots(figsize=(14, 3.5 * len(rows)))
    ax.imshow(np.concatenate(rows, axis=0))
    ax.set_yticks([ROW // 2 * (i + 0.5) for i in range(len(rows))])
    ax.set_yticklabels([p.stem.replace('step_', '') for p in grids])
    ax.set_xticks([])
    ax.set_title('строка = шаг обучения')
    plt.show()

---
## Модель

EMA-веса берутся из `ema_*.pt` (каждые 5000 шагов) или из `ckpt/last.ckpt` (последний чекпоинт Lightning, в том числе финальный).
Латенты модели живут с дисперсией 1, перед декодером они делятся на `scale`, как в обучении.

In [ ]:
vae = AutoencoderKL.from_pretrained(cfg['vae'], local_files_only=True, low_cpu_mem_usage=False)
vae = vae.to(DEVICE).eval().requires_grad_(False)
SCALE = cfg['scale']
sde = VPSDE(pred=cfg['pred'])
CMAP = colormaps['RdBu_r']


def load_ema(path):
    """EMA-веса U-Net из ema_*.pt или из ckpt/last.ckpt Lightning. Возвращает модель и шаг обучения."""
    ck = torch.load(path, map_location='cpu', weights_only=False, mmap=True)
    if 'ema' in ck:
        sd, step = ck['ema'], ck['step']
    else:
        sd, step = {k[4:]: v for k, v in ck['state_dict'].items() if k.startswith('ema.')}, ck['global_step']
    model = UNet(img_size=64, in_ch=4, base=cfg['base'], mults=[int(v) for v in cfg['mults'].split(',')],
                 num_res=cfg['num_res'], attn_res=[int(v) for v in cfg['attn_res'].split(',')],
                 dropout=cfg['dropout'], norm_groups=32,
                 head_ch=cfg.get('head_ch') or None, qk_norm=cfg.get('qk_norm', False))   # в ldm_v1 их еще не было
    model.load_state_dict(sd)
    return model.to(DEVICE).eval().requires_grad_(False), step


@torch.no_grad()
def decode(z, chunk=8):
    """Латенты модели (B, 4, 64, 64) -> картинки uint8 (B, 3, 512, 512) на CPU."""
    out = []
    for i in range(0, len(z), chunk):
        x = vae.decode(z[i:i + chunk].to(DEVICE).float() / SCALE).sample
        out.append(((x.clamp(-1, 1) + 1) * 127.5).round().byte().cpu())
    return torch.cat(out)


def to_np(img):
    """(3, H, W) uint8 -> (H, W, 3) для imshow."""
    return img.permute(1, 2, 0).numpy()


def latent_tile(z, lim=3.0):
    """(4, 64, 64) -> (128, 128, 3) в [0, 1]: четыре канала латента квадратом 2x2,
    синий ниже нуля, красный выше, насыщение на +-lim."""
    z = z.float()
    m = torch.cat([torch.cat([z[0], z[1]], 1), torch.cat([z[2], z[3]], 1)], 0).numpy()
    return CMAP((m / lim + 1) / 2)[..., :3]


@torch.no_grad()
def generate(model, n, seed=0, method='heun', steps=50, rho=None, n_keep=None):
    """
    Те же шаги, что sample() из train.py, но с историей.
      method : 'heun' (ОДУ, два вызова сети на шаг), 'euler' (ОДУ) или 'sde' (Эйлер-Маруяма)
      rho    : сгущение сетки по t к нулю, по умолчанию 2 для ОДУ и 1 для SDE, как в обучении
      n_keep : сколько шагов, равномерно, сохранить в историю, по умолчанию все
    Возвращает финальные латенты (n, 4, 64, 64) на CPU и history: список словарей с номером шага i, t, sigma,
    x (текущий зашумленный латент x_t) и x0 (догадка модели о финальном латенте), в float16 на CPU.
    """
    rho = rho if rho is not None else (1.0 if method == 'sde' else 2.0)
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    x = torch.randn(n, 4, 64, 64, generator=g, device=DEVICE)
    ts = sde.t_min + (1.0 - sde.t_min) * np.linspace(1.0, 0.0, steps + 1) ** rho
    keep = set(np.linspace(0, steps, min(n_keep or steps + 1, steps + 1)).round().astype(int).tolist())

    def eps_at(x, t):
        tt = torch.full((n,), float(t), device=DEVICE)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            e = model(x, sde.c(tt))
        return sde.to_eps(e.float(), x, tt)

    def drift(x, t, e, k=1.0):
        # правая часть при движении назад: x <- x + h * drift;  k = 1 для ОДУ, k = 2 для SDE
        tt = torch.tensor(float(t), device=DEVICE)
        return 0.5 * sde.beta(tt) * (x - k * e / sde.sigma(tt))

    history = []
    for i in range(steps + 1):
        t = ts[i]
        tt = torch.tensor(float(t), device=DEVICE)
        e = eps_at(x, t)
        x0 = (x - sde.sigma(tt) * e) / sde.m(tt)
        if i in keep:
            history.append(dict(i=i, t=float(t), sigma=float(sde.sigma(tt)), x=x.half().cpu(), x0=x0.half().cpu()))
        if i == steps:
            break
        h = t - ts[i + 1]
        if method == 'heun':
            d = drift(x, t, e)
            x_e = x + h * d
            x = x + 0.5 * h * (d + drift(x_e, ts[i + 1], eps_at(x_e, ts[i + 1])))
        elif method == 'euler':
            x = x + h * drift(x, t, e)
        elif method == 'sde':
            x = x + h * drift(x, t, e, k=2.0)
            if i < steps - 1:
                x = x + math.sqrt(float(sde.beta(float(t))) * h) * torch.randn(x.shape, generator=g, device=DEVICE)
        else:
            raise ValueError(method)
    return x0.float().cpu(), history

## Генерация

In [ ]:
# ------------------------------------------------------------- параметры
CKPT = 'ckpt/last.ckpt'          # или 'ema_0025000.pt'
SEED = 0
BATCH = 16
METHOD = 'heun'                  # 'heun', 'euler' или 'sde'
STEPS = 50                       # для 'sde' берите 1000
N_KEEP = 51                      # шагов в истории для разбора ниже, для sde 1000 хватит 101
COLS = 4

# ------------------------------------------------------------- генерация
model, step = load_ema(RUN_DIR / CKPT)
final, history = generate(model, BATCH, seed=SEED, method=METHOD, steps=STEPS, n_keep=N_KEEP)
imgs = decode(final)

# каждая картинка в отдельный файл, 512x512
name = f'step{step:07d}_{METHOD}{STEPS}_seed{SEED}'
out_dir = RUN_DIR / 'generated' / name
out_dir.mkdir(parents=True, exist_ok=True)
for i, im in enumerate(imgs):
    Image.fromarray(to_np(im)).save(out_dir / f'{i:03d}.png')
print(f'шаг обучения {step}, {BATCH} картинок сохранено в {out_dir}')

# ------------------------------------------------------------- рисунок
rows = math.ceil(BATCH / COLS)
fig, axs = plt.subplots(rows, COLS, figsize=(4.5 * COLS, 4.7 * rows), squeeze=False)
for ax in axs.flat:
    ax.axis('off')
for i, im in enumerate(imgs):
    axs.flat[i].imshow(to_np(im))
    axs.flat[i].set_title(str(i))
plt.suptitle(f'{name}', fontsize=15)
plt.tight_layout()
plt.show()

## Разбор по шагам расшумления

Четыре строки для одной картинки:

1. `x_t` в латенте: четыре канала квадратом 2x2, синий ниже нуля, красный выше.
2. `x_t` через декодер VAE. Декодер не обучен на шумных латентах, поэтому в начале это не картинка, а его реакция на шум.
3. `x̂_0` в латенте: что модель на этом шаге считает финальным латентом, `x̂_0 = (x_t - σ ε̂) / m`.
4. `x̂_0` через декодер: как выглядит эта догадка.

Следующая ячейка декодирует все шаги истории для выбранной картинки, на них же работает ползунок ниже.

In [ ]:
IDX = 0          # номер картинки из батча, он подписан над ней на рисунке выше
N_FRAMES = 8     # кадров на рисунке

dec_x = decode(torch.stack([fr['x'][IDX] for fr in history]))
dec_x0 = decode(torch.stack([fr['x0'][IDX] for fr in history]))

pick = np.linspace(0, len(history) - 1, N_FRAMES).round().astype(int)
fig, axs = plt.subplots(4, N_FRAMES, figsize=(2.6 * N_FRAMES, 11.5))
for ax in axs.flat:
    ax.set_xticks([])
    ax.set_yticks([])
for c, j in enumerate(pick):
    fr = history[j]
    axs[0, c].imshow(latent_tile(fr['x'][IDX]), interpolation='nearest')
    axs[0, c].set_title(f"шаг {fr['i']}, t = {fr['t']:.3f}\n$\\sigma$ = {fr['sigma']:.3f}", fontsize=10)
    axs[1, c].imshow(to_np(dec_x[j]))
    axs[2, c].imshow(latent_tile(fr['x0'][IDX]), interpolation='nearest')
    axs[3, c].imshow(to_np(dec_x0[j]))
for r, label in enumerate(['$x_t$, латент', '$x_t$, VAE', '$\\hat x_0$, латент', '$\\hat x_0$, VAE']):
    axs[r, 0].set_ylabel(label, fontsize=12)
plt.suptitle(f'картинка {IDX}, {name}', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
SIDE = 320    # сторона одной панели на экране, в пикселях


def png(arr):
    """(H, W, 3) uint8 или float в [0, 1] -> байты png для widgets.Image."""
    if arr.dtype != np.uint8:
        arr = (arr * 255).round().astype(np.uint8)
    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, 'png')
    return buf.getvalue()


# латент увеличиваем без сглаживания, чтобы было видно отдельные ячейки 64x64
big = lambda a: np.kron(a, np.ones((4, 4, 1)))
panels = [[png(big(latent_tile(fr['x'][IDX]))) for fr in history],
          [png(to_np(im)) for im in dec_x],
          [png(big(latent_tile(fr['x0'][IDX]))) for fr in history],
          [png(to_np(im)) for im in dec_x0]]
titles = ['x_t, латент', 'x_t, VAE', 'x̂_0, латент', 'x̂_0, VAE']
last = len(history) - 1

box = widgets.Layout(width=f'{SIDE}px', height=f'{SIDE}px')
imgs_w = [widgets.Image(value=p[0], format='png', layout=box) for p in panels]
cols = [widgets.VBox([widgets.HTML(f'<b>{t}</b>'), w]) for t, w in zip(titles, imgs_w)]
info = widgets.HTML()

# Play двигает ползунок сам: вся история примерно за 10 с, быстрее 50 мс на кадр не идем
play = widgets.Play(min=0, max=last, interval=max(50, 10_000 // len(history)), description='запуск')
slider = widgets.IntSlider(min=0, max=last, description='шаг', layout=widgets.Layout(width=f'{4 * SIDE - 80}px'))
widgets.jslink((play, 'value'), (slider, 'value'))


def show_frame(k):
    for w, p in zip(imgs_w, panels):
        w.value = p[k]
    fr = history[k]
    info.value = (f"<b>картинка {IDX}, {name}</b>: шаг {fr['i']} из {STEPS}, t = {fr['t']:.3f}, "
                  f"&sigma; = {fr['sigma']:.3f}, std x_t = {fr['x'][IDX].float().std():.2f}")


slider.observe(lambda change: show_frame(change['new']), names='value')
show_frame(0)
display(widgets.VBox([widgets.HBox([play, slider]), info, widgets.HBox(cols)]))

---
## Как менялась модель по ходу обучения

Одни и те же шумы через каждый сохраненный чекпоинт, Heun 50 шагов. Строка = шаг обучения.

In [ ]:
N_CMP = 6
SEED_CMP = 0

ckpts = sorted(RUN_DIR.glob('ema_*.pt')) + [RUN_DIR / 'ckpt' / 'last.ckpt']
results = []
for p in ckpts:
    m, st = load_ema(p)
    fin, _ = generate(m, N_CMP, seed=SEED_CMP, method='heun', steps=50, n_keep=1)
    results.append((st, decode(fin)))
    del m
    torch.cuda.empty_cache()

fig, axs = plt.subplots(len(results), N_CMP, figsize=(2.8 * N_CMP, 2.9 * len(results)), squeeze=False)
for r, (st, ims) in enumerate(results):
    for c in range(N_CMP):
        axs[r, c].imshow(to_np(ims[c]))
        axs[r, c].set_xticks([])
        axs[r, c].set_yticks([])
    axs[r, 0].set_ylabel(f'шаг {st}', fontsize=12)
plt.tight_layout()
plt.show()